##  필요한 패키지 설치

In [1]:
%pip install transformers datasets accelerate trl peft langchain python-dotenv -qU


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Dataset Loading

In [2]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

print(load_dotenv('env'))

login(os.getenv('HUGGINGFACE_API_KEY'))

True


In [5]:
from datasets import load_dataset

data_id = "kgmyh/naver_economy_news_stock_instruct_dataset-100"

dataset = load_dataset(data_id)

train_set = dataset['train']
test_set = dataset['test']

# sLLM Model Load

## Base Model Load
- 한국어를 충분히 학습한 모델을 선택한다.

- kakao의 kanana 모델을 base 모델로 파인튜닝을 진행한다.
  - https://tech.kakao.com/posts/660
  - https://github.com/kakao/kanana
  - 2.1b 모델을 open source로 공개함. 


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


model_id = "kakaocorp/kanana-1.5-2.1b-instruct-2505" # JSON 포멧은 잘 지켜서 나오는데 회사나 이런게 없는 회사가 나오고 한다.

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", 
    torch_dtype=torch.bfloat16, # 양자화 32비트 짜리를 bfloat16으로
    attn_implementation="eager" 
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/66.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

### 사용해보기

In [16]:
####################################################################################################################################################
#  입력 프롬프트 생성
#
#  - kanana, Llama 동일한 프롬프트 형식 (자세한 형식은 아래 있다.)
#    - Instruction 모델이 학습할 때 사용한 prompt 형식에 맞춰 입력데이터를 변환을 해야 한다.
#  - `apply_chat_template`: 입력 형태 기본: {"role":"역할", "content":"content"}  이 형식으로 입력을 하면 모델의 입력형식으로 변환해 주는 메소드.
####################################################################################################################################################
content = "오늘 서울 날씨 어때요?"
content = "오늘 서울 날씨 어때요? 모르면 모른다고 답하세요."
message = [  
        {"role": "system", "content": "당신은 인공지는 날씨 예보관입니다."},
        {"role": "user", "content": content},
]
prompt = tokenizer.apply_chat_template(
    message,
    tokenize=False,            # 토큰 id로 반환할 지 여부 (여기서는 텍스트 확인을 위해 False)
    add_generation_prompt=True # 마지막에 assistant  role에 대한 역할 식별자를 붙일 지 여부. (마지막에 `답변:` 이렇게 넣는 것과 같다.)
)
print(prompt)
####################################3
# LLM 입장에서는 string 값으로 입력을 받아서 확률 기반으로 뒤에 문장을 생성하는 애
# 얘를 질의응답으로 하려면 질문이 들어왔을 때 뭐라고 할지 답변을 학습하기 때문에 어쨌든 문장을 생성하는 애로 볼 수 있다
# 뭔 소리냐 이게,,,
# LLM 모델에서 줄 때는 string 값으로 형식화시켜줘야 한다.
# role과 내용을 모두 분리해서 string 값으로 입력을 해줘야 llm이 "답변"을 생성할 수 있는데 이 역할을 chat_template이 한다 
# => chat_template이 실제 llm이 학습할 수 있는 형태로 바꿔준다


<|begin_of_text|><|start_header_id|>system<|end_header_id|>


<|eot_id|><|start_header_id|>system<|end_header_id|>

당신은 인공지는 날씨 예보관입니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

오늘 서울 날씨 어때요? 모르면 모른다고 답하세요.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [8]:
tokenizer.chat_template # chat_tempalte의 모든 문법, 다 알 필요는 없다!

'{# version=v3-llama3.1 #}{%- macro append_new_param_info(param_declaration, comment_info, examples_info, depth) -%}\n    {%- set offset = "" -%}\n    {%- if depth >= 1 -%}\n        {%- set offset = "    " * depth -%}\n    {%- endif -%}\n    {%- if comment_info != "<|NONE|>" -%}\n        {{ "\\n" + offset + comment_info }}\n        {%- if examples_info | length > 0 -%}\n            {# Append each example info #}\n            {%- for example in examples_info -%}\n                {{ "\\n" + offset + "// " + example|string|replace("\'", \'"\') }}\n            {%- endfor -%}\n        {%- endif -%}\n    {%- endif -%}\n    {{ "\\n" + offset + param_declaration }}\n{%- endmacro -%}\n\n{%- macro convert_data_type(param_type) -%}\n    {%- if param_type == "integer" or param_type == "float" -%}\n        {{ "number" }}\n    {%- else -%}\n        {{ param_type }}\n    {%- endif -%}\n{%- endmacro -%}\n\n{%- macro get_param_type(param) -%}\n    {%- set param_type = "any" -%}\n\n    {%- if "type" in 

In [17]:
########################
# 토큰화
########################

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(inputs)

{'input_ids': tensor([[128000, 128000, 128006,   9125, 128007,   1432, 128009, 128006,   9125,
         128007,    271,  65895,  83628,  34804,  59777, 103896,  16969, 105605,
         107497,  96717,  42771, 101106,  80052,     13, 128009, 128006,    882,
         128007,    271,  58368, 105622, 106010, 105605, 107497, 101139, 106745,
          36811,     30, 117215,  33390,  55170,  84065, 105954, 108386,  92245,
             13, 128009, 128006,  78191, 128007,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}


In [10]:
inputs.keys()
# ['input_ids'] : text를 token_id로 변환한 값들
# 'attention_mask' : text와 padding을 구분하는 값 1: text, 0: padding
# 모델마다 필요한 값들이 다르다 -> 그래서 모델과 tokenizer를 맞춰줘야 하는 것

dict_keys(['input_ids', 'attention_mask'])

In [25]:
################### 
# 생성
###################

import torch
with torch.no_grad():
    output = model.generate(
        **inputs,            # input_ids와 attention_mask를 "풀어서" 넣어줌
        max_new_tokens=100,  # 답변 토큰 수 제한
        do_sample=True,      # 확률 붙포에 따라 다음 토큰을 무작위로 선택(True - top-k, temperature와 같이 쓰인다.), False: 가장 높은 확률의 토큰만 선택
        top_p=0.95,          # 뉴클리어스(nucleus) 샘플링의 누적 확률 임계값 - 다음에 올 확률 순으로 토큰들 정렬 → 누적 → 0.95(지정한값) 도달 시점까지 상위토큰만 남긴다. ->  집합에서 샘플링. 낮으면(예: 0.5) 창의성↓, 너무 높으면(1.0) 창의성↑
        temperature=0.8,     # 토큰의 다양성을 지정. 낮을수록(0에 가까울수록) 높은 확률의 토큰을 더욱 선택. 낮으면(예: 0.5) 창의성↓, 너무 높으면(1.0) 창의성↑
        pad_token_id=tokenizer.eos_token_id,  # 패딩 토큰 ID. 대다수 LLM은 별도의 pad_token이 없이 eos_token(end of sequence token)으로 대신함
    )

In [12]:
output
# token id로 나오는 것 확인
# 실제 문자로 바꿔줘야 함 -> decode

tensor([[128000, 128000, 128006,   9125, 128007,   1432, 128009, 128006,   9125,
         128007,    271,  65895,  83628,  34804,  59777, 103896,  16969, 105605,
         107497,  96717,  42771, 101106,  80052,     13, 128009, 128006,    882,
         128007,    271,  58368, 105622, 106010, 105605, 107497, 101139, 106745,
          36811,     30, 117215,  33390,  55170,  84065, 105954, 108386,  92245,
             13, 128009, 128006,  78191, 128007,    271,  35859,  80979, 102326,
         108076, 105605, 107497,  61139,  18918,  74959,  72043,  80052,     13,
         109562,  30426,  73653, 123118, 101103,  56773,  51402,     13,  19124,
         105922,  20565, 107022,    167,  47318,  29726, 107205,  62060,  17835,
         106010,  21028, 111530, 105605, 107497,  81673,  96717,  42771,  18918,
         116023, 124365, 115284,     13, 128009]], device='cuda:0')

In [19]:
print(tokenizer.decode(output[0]))
# 우리가 입력해준 것도 다 decode 해주는 걸 확인
# 사실 우리가 입력해준 건 응답에 필요없다 -> <|eot_id|><|start_header_id|>assistant<|end_header_id|> 뒤부터 필요함

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>


<|eot_id|><|start_header_id|>system<|end_header_id|>

당신은 인공지는 날씨 예보관입니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

오늘 서울 날씨 어때요? 모르면 모른다고 답하세요.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

오늘 서울의 날씨를 확인해보니, 현재 구름이 조금 끼어 있고 기온은 15°C 정도입니다. 강수 확률은 20%로, 비가 올 가능성은 낮은 편입니다. 외출 시 가벼운 옷차림이 적절합니다.<|eot_id|>


In [15]:
inputs['input_ids'].shape # 입력 token 크기
# torch.Size([1, 51])는 
# 입력이 하나인데 토큰으로 바꿨더니 51개다.
# 그래서 output[0][inputs["input_ids"].shape[-1]:] 이 의미는
# inputs["input_ids"].shape[-1] 이거 대신 51을 넣어서 슬라이싱 했다~ => 우리가 입력한 input_ids의 뺐다~ 

torch.Size([1, 51])

In [20]:
response = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True) #  skip_special_tokens=True는 special token 무시하란 의미
print(response)

오늘 서울의 날씨를 확인해보니, 현재 구름이 조금 끼어 있고 기온은 15°C 정도입니다. 강수 확률은 20%로, 비가 올 가능성은 낮은 편입니다. 외출 시 가벼운 옷차림이 적절합니다.


# 학습 전에 추론 시켜 성능 확인

### 추론용 프롬프트 만들기

In [22]:
######################################
#  System 프롬프트
######################################

system_prompt = '''# Instruction
당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.
사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.

## 분석 기준
1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.
2. 영향을 준다면 다음 항목을 출력하세요.
   - `"is_stock_related": true`
   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**
   - 뉴스에 **부정적** 영향을 받는 **회사이름들**
   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**
     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.
     - `None`, 유추, 추정, 일반 논평 금지합니다.
   - 뉴스 요약 (3줄 이내)
3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.
   - `"is_stock_related": false`
   - 뉴스 요약 (3줄 이내)

# 출력 지시사항 (Output Indicator)
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"is_stock_related": {"description": "한국 주식과 관련있는 뉴스인지 여부", "title": "Is Stock Related", "type": "boolean"}, "positive_stocks": {"description": "뉴스기사에 긍정적인 영향을 받는 회사들의 이름들.", "items": {"type": "string"}, "title": "Positive Stocks", "type": "array"}, "positive_reason": {"description": "뉴스내용 중 positive_stocks에 있는 각 회사들에 긍정적 영향을 주는 내용. {\"회사이름\":\"긍정적인 이유\"}", "items": {"additionalProperties": {"type": "string"}, "type": "object"}, "title": "Positive Reason", "type": "array"}, "negative_stocks": {"description": "뉴스기사에 부정적인 영향을 받는 회사들의 이름들.", "items": {"type": "string"}, "title": "Negative Stocks", "type": "array"}, "negative_reason": {"description": "뉴스내용 중 negative_stocks에 있는 각 회사들에 부정적 영향을 주는 내용. {\"회사이름\":\"부정적인 이유\"}", "items": {"additionalProperties": {"type": "string"}, "type": "object"}, "title": "Negative Reason", "type": "array"}, "summary": {"description": "뉴스기사 요약", "title": "Summary", "type": "string"}}, "required": ["is_stock_related", "positive_stocks", "positive_reason", "negative_stocks", "negative_reason", "summary"]}
```

## 출력 조건:
- 뉴스에 영향을 받은 회사들은 **반드시 한국 증시에 상장된 종목** 이어야 합니다.
- 뉴스에 있는 내용만 출력결과에 포함시킵니다.
- 긍정/부정 종목은 실제 뉴스기사에 영향을 받는 회사들만 포함하세요.
- 모든 문자열은 큰따옴표(`"`)로 감쌉니다.
- 문자열 안에 따옴표가 필요하면 작은따옴표(`'`)를 사용합니다.
- 모든 키(Key)는 출력 지시사항에 명시된 property들과 정확히 일치해야 합니다.
- `"positive_reasons"` 및 `"negative_reasons"`의 값은 `None`이 될 수 없습니다.
- json format을 잘 지켜 응답데이터를 만듭니다. 배열이나 object의 마지막 항목 뒤에 `,` 를 붙이지 마세요.
- 오직 유효한 JSON 문자열(UTF-8, RFC8259 준수)만 출력합니다.
- 절대 다른 텍스트, 주석, 설명, 코드 블록 표기(```), 또는 따옴표 외의 문자열을 추가하면 안 됩니다.

## 출력 예시 (Examples)

### 뉴스가 특정 주식종목들에 **긍정적 영향이 주는 경우**:
{'is_stock_related': True,
 'negative_reasons': [],
 'negative_stocks': [],
 'positive_reasons': [{'세라젬': '루게릭병 환우 지원 캠페인 후원과 의료가전 지원 등 사회공헌활동을 통해 기업 이미지와 브랜드 가치가 긍정적으로 부각됨'}],
 'positive_stocks': ['세라젬'],
 'summary': '세라젬이 루게릭병 환우를 위한 아이스버킷 챌린지 런 행사를 후원하며 의료가전과 건강기능식품 등을 지원했다. 캠페인은 루게릭병 환우 지원과 기부 문화 확산을 목표로 한다. 세라젬은 다양한 사회공헌활동을 지속하고 있다.'
}

### 뉴스의 내용이 특정 주식종목들에 **부정적 영향이 주는 경우**:
{
    "is_stock_related": true,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [
        "포스코",
        "현대제철"
    ],
    "negative_reasons": [
        {"포스코": "정부가 수입규제국 조사에 적극 대응하고 비관세장벽 해소를 위해 민관 협력 강화 방침을 밝혀 철강 분야에서 수출 피해 최소화 기대"},
        {"현대제철": "철강·금속 품목에 대한 수입규제 대응 강화로 불합리한 무역제한 조치 개선 가능성이 높아져 수출 환경 개선 기대"}
    ],
    "summary": "산업부는 수입 규제국의 조사에 대응하고 비관세장벽 해소를 위한 협의를 진행했다. 규제 대상 국가는 26개국, 건수는 199건에 달한다."
}

### **뉴스기사가 주식 종목과 관련 없는 경우**:
{
    "is_stock_related": false,
    "positive_stocks": [],
    "positive_reasons": [],
    "negative_stocks": [],
    "negative_reasons": [],
    "summary": "정황근 농림축산식품부 장관이 단순가공식품 부가가치세 면제 시행 상황을 점검했다. 된장, 고추장 코너를 방문하며 현장을 살폈다."
}'''


In [23]:
################################################################
# 추론할 1개의 샘플 데이터 생성
# 뉴스 기사 제목 + "\n\n" + 뉴스 기사 내용
################################################################

idx = 10

sample = dataset['train'][idx]
user_input = sample['title']+"\n\n"+sample['document']
prompt = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content": user_input}
]
prompt

[{'role': 'system',
  'content': '# Instruction\n당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.\n사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.\n\n## 분석 기준\n1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.\n2. 영향을 준다면 다음 항목을 출력하세요.\n   - `"is_stock_related": true`\n   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**\n   - 뉴스에 **부정적** 영향을 받는 **회사이름들**\n   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**\n     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.\n     - `None`, 유추, 추정, 일반 논평 금지합니다.\n   - 뉴스 요약 (3줄 이내)\n3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.\n   - `"is_stock_related": false`\n   - 뉴스 요약 (3줄 이내)\n\n# 출력 지시사항 (Output Indicator)\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\n

In [26]:
#########################################################
# Pipeline을 이용해 실행
#  - task: text-generation
#  - pipeline은 (role base) chat format 으로 입력한다. 
#########################################################
from transformers import pipeline

pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer )
res = pipe(prompt,  max_new_tokens=200)
# _chat_template을 안 썼어 왜?
# pipeline을 썼기 때문에 전체적인 process를 다 해주기 때문에
# 직접 추론을 해줄 때는 chat_template 형태로 바꿔줘야하지만, pipeline에서는 안 해줘도 된다

Device set to use cuda:0


In [27]:
print(res)

[{'generated_text': [{'role': 'system', 'content': '# Instruction\n당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.\n사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.\n\n## 분석 기준\n1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.\n2. 영향을 준다면 다음 항목을 출력하세요.\n   - `"is_stock_related": true`\n   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**\n   - 뉴스에 **부정적** 영향을 받는 **회사이름들**\n   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**\n     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.\n     - `None`, 유추, 추정, 일반 논평 금지합니다.\n   - 뉴스 요약 (3줄 이내)\n3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.\n   - `"is_stock_related": false`\n   - 뉴스 요약 (3줄 이내)\n\n# 출력 지시사항 (Output Indicator)\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "req

In [31]:
len(res[0]['generated_text'])
response = res[0]['generated_text'][-1] # 우리가 원하는 응답만 뽑음
response['content']
# prompt에 맞춰서 어느 정도 나오는 걸 확인

'```json\n{\n  "is_stock_related": true,\n  "negative_stocks": [\n    "세라젬"\n  ],\n  "negative_reasons": [\n    {\n      "세라젬": "이준석 국민의힘 대표 징계 심의 착수 및 이번주 주요 일정 언급으로 이 대표가 소속된 세라젬 등 관련 기업의 주가에 불확실성 및 정치적 리스크가 확대됨"\n    }\n  ],\n  "positive_stocks": [\n    "포스코", "현대제철"\n  ],\n  "positive_reasons": [\n    {\n      "포스코": "일제강점기 강제동원 피해자 배상 민관협의회 출범 소식으로, 철강·금속 관련 기업인 포스코의 주가에 긍정적 영향을 미칠 것으로 전망됨. 협의회가 배상 해법 논의의 출'

# 파인 튜닝 

## 데이터셋 만들기

### 프롬프트 생성
- LLM 모델은 학습할때 사용한 프롬프트 형식이 있다.
    - 모델 마다 형식이 다르기 때문에 확인이 필요하다.
    - 모델이 사용한 tokenizer의 chat_template 속성을 이용해 조회할 수 있다.
      - `tokenizer.chat_template`
      - `jija2` 템플릿 엔진 문법으로 작성됨.
- 학습데이터를 LLM의 프롬프트 형식으로 생성한다.

### Llama 모델의 chat template 형식
```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
[시스템 역할 지침]<|eot_id|>
<|start_header_id|>user<|end_header_id|>
[유저 질문]<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
[모델의 답변]<|eot_id|>
```
=> system, user, assistant 모두 학습이 필요하다 그래서 입력 형식은 3개로 다 넣어주고,  
어느 부분이 답변이라는 masking을 넣어준다  
=> 이를 위해선 data를 chat template 형식으로 바꿔줘야 한다.

- `<|begin_of_text|>`: 시퀀스의 시작을 나타내는 토큰.
- `<|start_header_id|>ROLE<|end_header_id|>`: Role(발화자) 지정 - system | user | assistant
- `메세지 내용<|eot_id|>`:  Role 메세지. 시퀀스 종료를 나타내는 토큰(`<|eot_id|>`)

#### 예
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
당신은 인공지는 날씨 예보관입니다.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
오늘 서울 날씨 어때요?<|eot_id|>
```
> ### Gemma format chat template 형식
> 
> ```xml
> <bos><start_of_turn>Role
> {사용자 입력-input}<end_of_turn>
> <start_of_turn>model
> {AI 답변-label}<end_of_turn>
> ```
> -	`<bos>`: 시퀀스의 시작을 나타내는 토큰.
> -	`<start_of_turn>`: 각 대화의 시작을 나타낸다.
> -	`Role`: Role(발화자) 지정 - **user** | **model** 
>   - gemma는 user와 model 두가지 role을 이용해 instruction 모델을 학습함.
>   - gemma 는 system role을 사용하지 않는다. 그래서 **system prompt는 user role의 content에 넣어준다.**
> - `메세지`
>   - Role 다음 줄에 이어서 메세지를 입력한다.
> - `<end_of_turn>`: 대화의 끝을 나타냅니다.
> - 예
> ```xml
> <bos><start_of_turn>user
> AI에 대해 설명해주세요.<end_of_turn>
> <start_of_turn>model
> AI은 컴퓨터 시스템이 인간의 지능적 기능을 모방하여 데이터를 처리하고 의사결정을 수행하는 기술입니다.<end_of_turn>
> ```

> ### Alapaca format chat template
> ```
> ### Instruction:
> [시스템 역할 지침]
> 
> [유저 질문]
> 
> ### Response:
> [모델의 답변]
> ```
> - `### Instruction`: - 사용자 입력/질문
> - `### Response`: - 모델의 답변
> - 특수 토큰 대신 마크다운 스타일의 헤더 사용
> - 보통 맨 앞에 표준 instruction 문구 포함


### InputPromptCreator(프롬프트 생성 클래스) 정의

- `create_pipeline_prompt()`:
    - 뉴스기사 제목, 뉴스기사내용을 받아서 파이프라인에 입력할 chat 형식 프롬프트생성한다
    - dict 안에 list 형태로 return
- `create_generate_prompt:()`:
    - 뉴스기사 제목, 뉴스기사내용을 받아서  모델의 자체 chat 형식의 프롬프트를 생성한다.  
여기까지 "추론"에 필요한 애들  

- `create_train_prompt()`:
    - Dataset의 개별 데이터를 입력받아서 모델 학습을 위한 chat 프롬프트생성
    - label을 붙여야해,,!

In [32]:
####################################################################
# 학습 용 프롬프트 생성 함수
####################################################################


from textwrap import dedent
class InputPromptCreator:
    """모델별 형식에 맞춰 입력 프롬프트를 생성한다.
    """
    def __init__(self, tokenizer=None):
        """
        모델의 chat template을 제공하는 tokenizer와 system 프롬프트를 받아서 초기화
        """
        self.tokenizer = tokenizer
        self.system_prompt = dedent('''
        # Instruction
        당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.
        사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.
        
        ## 분석 기준
        1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.
        2. 영향을 준다면 다음 항목을 출력하세요.
           - `"is_stock_related": true`
           - 뉴스에 **긍정적** 영향을 받는 **회사이름들**
           - 뉴스에 **부정적** 영향을 받는 **회사이름들**
           - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**
             - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.
             - `None`, 유추, 추정, 일반 논평 금지합니다.
           - 뉴스 요약 (3줄 이내)
        3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.
           - `"is_stock_related": false`
           - 뉴스 요약 (3줄 이내)
        
        # 출력 지시사항 (Output Indicator)
        The output should be formatted as a JSON instance that conforms to the JSON schema below.
        
        As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
        the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.
        
        Here is the output schema:
        ```
        {"properties": {"is_stock_related": {"description": "한국 주식과 관련있는 뉴스인지 여부", "title": "Is Stock Related", "type": "boolean"}, "positive_stocks": {"description": "뉴스기사에 긍정적인 영향을 받는 회사들의 이름들.", "items": {"type": "string"}, "title": "Positive Stocks", "type": "array"}, "positive_reason": {"description": "뉴스내용 중 positive_stocks에 있는 각 회사들에 긍정적 영향을 주는 내용. {\"회사이름\":\"긍정적인 이유\"}", "items": {"additionalProperties": {"type": "string"}, "type": "object"}, "title": "Positive Reason", "type": "array"}, "negative_stocks": {"description": "뉴스기사에 부정적인 영향을 받는 회사들의 이름들.", "items": {"type": "string"}, "title": "Negative Stocks", "type": "array"}, "negative_reason": {"description": "뉴스내용 중 negative_stocks에 있는 각 회사들에 부정적 영향을 주는 내용. {\"회사이름\":\"부정적인 이유\"}", "items": {"additionalProperties": {"type": "string"}, "type": "object"}, "title": "Negative Reason", "type": "array"}, "summary": {"description": "뉴스기사 요약", "title": "Summary", "type": "string"}}, "required": ["is_stock_related", "positive_stocks", "positive_reason", "negative_stocks", "negative_reason", "summary"]}
        ```
        
        ## 출력 조건:
        - 뉴스에 영향을 받은 회사들은 **반드시 한국 증시에 상장된 종목** 이어야 합니다.
        - 뉴스에 있는 내용만 출력결과에 포함시킵니다.
        - 긍정/부정 종목은 실제 뉴스기사에 영향을 받는 회사들만 포함하세요.
        - 모든 문자열은 큰따옴표(`"`)로 감쌉니다.
        - 문자열 안에 따옴표가 필요하면 작은따옴표(`'`)를 사용합니다.
        - 모든 키(Key)는 출력 지시사항에 명시된 property들과 정확히 일치해야 합니다.
        - `"positive_reasons"` 및 `"negative_reasons"`의 값은 `None`이 될 수 없습니다.
        - json format을 잘 지켜 응답데이터를 만듭니다. 배열이나 object의 마지막 항목 뒤에 `,` 를 붙이지 마세요.
        - 오직 유효한 JSON 문자열(UTF-8, RFC8259 준수)만 출력합니다.
        - 절대 다른 텍스트, 주석, 설명, 코드 블록 표기(```), 또는 따옴표 외의 문자열을 추가하면 안 됩니다.
        
        ## 출력 예시 (Examples)
        
        ### 뉴스가 특정 주식종목들에 **긍정적 영향이 주는 경우**:
        {'is_stock_related': True,
         'negative_reasons': [],
         'negative_stocks': [],
         'positive_reasons': [{'세라젬': '루게릭병 환우 지원 캠페인 후원과 의료가전 지원 등 사회공헌활동을 통해 기업 이미지와 브랜드 가치가 긍정적으로 부각됨'}],
         'positive_stocks': ['세라젬'],
         'summary': '세라젬이 루게릭병 환우를 위한 아이스버킷 챌린지 런 행사를 후원하며 의료가전과 건강기능식품 등을 지원했다. 캠페인은 루게릭병 환우 지원과 기부 문화 확산을 목표로 한다. 세라젬은 다양한 사회공헌활동을 지속하고 있다.'
        }
        
        ### 뉴스의 내용이 특정 주식종목들에 **부정적 영향이 주는 경우**:
        {
            "is_stock_related": true,
            "positive_stocks": [],
            "positive_reasons": [],
            "negative_stocks": [
                "포스코",
                "현대제철"
            ],
            "negative_reasons": [
                {"포스코": "정부가 수입규제국 조사에 적극 대응하고 비관세장벽 해소를 위해 민관 협력 강화 방침을 밝혀 철강 분야에서 수출 피해 최소화 기대"},
                {"현대제철": "철강·금속 품목에 대한 수입규제 대응 강화로 불합리한 무역제한 조치 개선 가능성이 높아져 수출 환경 개선 기대"}
            ],
            "summary": "산업부는 수입 규제국의 조사에 대응하고 비관세장벽 해소를 위한 협의를 진행했다. 규제 대상 국가는 26개국, 건수는 199건에 달한다."
        }
        
        ### **뉴스기사가 주식 종목과 관련 없는 경우**:
        {
            "is_stock_related": false,
            "positive_stocks": [],
            "positive_reasons": [],
            "negative_stocks": [],
            "negative_reasons": [],
            "summary": "정황근 농림축산식품부 장관이 단순가공식품 부가가치세 면제 시행 상황을 점검했다. 된장, 고추장 코너를 방문하며 현장을 살폈다."
        }''')

    def create_pipeline_prompt(self, news_title:str, news_document:str) -> list[dict]:
        """파이프라인에 입력할 chat 형식 프롬프트생성한다.
        ```
        [
            {"role":"system", "content":시스템프롬프트},
            {"role":"user", "content": news_title+"\n\n"+news_document}
        ]
        ```
        """
        news = news_title+"\n\n"+news_document
        message = [
            {"role":"system", "content":self.system_prompt},
            {"role":"user", "content": news}
        ]
        return message

    def create_generate_prompt(self, news_title:str, news_document:str) -> str:
        """뉴스 제목과 내용을 받아서 tokenizer를 이용해 모델의 자체 chat 형식의 프롬프트를 생성한다.
        """
        if self.tokenizer is None:
            raise Exception("Tokenizer가 없습니다. generate_prompt를 사용하려면 모델의 tokenizer가 필요합니다.")
        message = self.create_pipeline_prompt(news_title, news_document)
        prompt =  tokenizer.apply_chat_template(
            message,
            tokenize=False,
            add_generation_prompt=True
        )
        return prompt

    def create_train_prompt(self, datapoint:dict) -> dict:
        """
        Dataset의 개별 데이터를 입력받아서 모델 학습을 위한 chat 프롬프트생성
        프롬프트 형식
        ```
        <|begin_of_text|>
        <|start_header_id|>system<|end_header_id|>
        {시스템 프롬프트}<|eot_id|>
        <|start_header_id|>user<|end_header_id|>
        {입력 - title+document}<|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        {답변 - Label}<|eot_id|>
        ```
        Args:
            datapoint (dict): 변환할 데이터
    
        Returns:
            str: 모델 학습을 위한 프롬프트. 
        """
        chat_template = dedent('''
            <|begin_of_text|>
            <|start_header_id|>system<|end_header_id|>
            {system_prompt}<|eot_id|>
            <|start_header_id|>user<|end_header_id|>
            {input_content}<|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            {label}<|eot_id|>''')
    
        content = datapoint['title']+"\n"+datapoint['document']
        prompt = chat_template.format(system_prompt=self.system_prompt, input_content=content, label=datapoint['label'])

        return {"train_prompt":prompt} # 학습할 때 이런 구조로 해야하기 때문에 return 값이 dictionary

In [33]:
prompt_creator = InputPromptCreator(tokenizer=tokenizer)
prompt_creator.create_pipeline_prompt('>>>>>>뉴스기사제목',' <<<<<<<뉴스내용')
# system prompt는 그대로인데
# 마지막에  {'role': 'user', 'content': '>>>>>>뉴스기사제목\n\n <<<<<<<뉴스내용'}] 이렇게 된 걸 확인

[{'role': 'system',
  'content': '\n# Instruction\n당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.\n사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.\n\n## 분석 기준\n1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.\n2. 영향을 준다면 다음 항목을 출력하세요.\n   - `"is_stock_related": true`\n   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**\n   - 뉴스에 **부정적** 영향을 받는 **회사이름들**\n   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**\n     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.\n     - `None`, 유추, 추정, 일반 논평 금지합니다.\n   - 뉴스 요약 (3줄 이내)\n3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.\n   - `"is_stock_related": false`\n   - 뉴스 요약 (3줄 이내)\n\n# 출력 지시사항 (Output Indicator)\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}

In [35]:
prompt_creator.create_train_prompt(train_set[0])
# 정답 label이 붙은 걸 확인

{'train_prompt': '\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\n\n# Instruction\n당신은 금융 뉴스의 핵심 내용을 요약해 설명하고, 뉴스가 특정 상장 종목에 미치는 긍정/부정 영향 여부, 이유, 근거 등을 분석하는 금융 분석 전문가입니다.\n사용자에 의해 입력된 뉴스 기사를 분석해서 **한국에 상장된 주식 종목에 영향을 주는지 판단**하고, Output Indicator에 제시된 기준에 따라 구조화된 JSON 형식으로 결과를 출력하세요.\n\n## 분석 기준\n1. 뉴스가 **한국 주식 종목에 영향을 주는지 판단**하세요.\n2. 영향을 준다면 다음 항목을 출력하세요.\n   - `"is_stock_related": true`\n   - 뉴스에 **긍정적** 영향을 받는 **회사이름들**\n   - 뉴스에 **부정적** 영향을 받는 **회사이름들**\n   - 뉴스가 각 회사에 **긍정적 또는 부정적 영향을 주는지 이유**\n     - 반드시 **뉴스기사에 언급된 내용 기반으로 작성한다.** 뉴스기사에 없는 내용을 꾸며서 임의로로 작성하지 않습니다.\n     - `None`, 유추, 추정, 일반 논평 금지합니다.\n   - 뉴스 요약 (3줄 이내)\n3. 뉴스가 한국 주식 종목에 영향을 주지 않는다면 다음 항목을 출력하세요.\n   - `"is_stock_related": false`\n   - 뉴스 요약 (3줄 이내)\n\n# 출력 지시사항 (Output Indicator)\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "i

In [ ]:
dataset에다 train_prompt라는 feature를 만들 것
원래 있던 features는 필요없으니 지워도 됨

### InputPromptCreator.create_train_prompt()를 이용해 입력 프롬프트 만들기

In [44]:
##########################
# Trainset
##########################

llama_input_creator = InputPromptCreator(tokenizer)
# Dataset.map(함수) -> 개별 데이터를 함수에 전달. 함수의 반환값({feature 이름:값}) 을 Dataset에 추가
trainset = train_set.map(llama_input_creator.create_train_prompt, remove_columns=list(train_set.features))
# dataset에다 train_prompt라는 feature를 만들 것
# 원래 있던 features는 필요없으니 지워도 됨
trainset

Dataset({
    features: ['train_prompt'],
    num_rows: 90
})

In [45]:
##########################
# Testset
##########################

testset = test_set.map(llama_input_creator.create_train_prompt, remove_columns=list(test_set.features))
testset

Dataset({
    features: ['train_prompt'],
    num_rows: 10
})

## 파인튜닝

### Data Collator

- 학습 도중 입력 데이터를 받아 전처리하는 함수
  - Dataset -> Data Collator함수 -> 모델
    - 데이터셋에서 모델에 전달되는 batch를 받아서 모델에 입력전에 해야하는 처리를 담당하는 함수(Callable).

- 구현할 내용
  - 모델 chat 형식의 문자열을 transformers 모델에 입력하기 위한 inputs를 만든다.
    
  ```json
  {
  "input_ids":입력 sequence의 토큰 ID들,
  "attention_mask":입력토큰과 padding구분,
  "labels": input_ids에서 답변부분 masking. 답변은 토큰ID 나머지는 -100으로 채운다. 
  }
  ```

In [46]:
import torch

max_seq_length = 8192

def collate_fn(batch):
    # batch: list[dict]
    #  dict - {"train_prompt":input text}
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }
    
    for prompt in batch:
        
        text = prompt['train_prompt'].strip()
        
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length, # max_length이하는 자른다. truncation=True
            padding=False,             # padding은 뒤에서 수동으로 처리할 것이기 때문에 padding처리하지 않는다.
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = [-100] * len(input_ids) #모델 응답 부분만 label로 지정: input_ids에서 "system/user 프롬프트는 -100", "label은 실제 토큰값" 으로 masking한다. 그것을 위해서 input_ids와 동일한 크기의 -100으로 채워진 리스트 생성한다.
                                         # -100인 이유: PyTorch의 CrossEntropyLoss(ignore_index=-100)가 loss계산시 무시하는 값이다.

        ########################################################
        # chat prompt에서 답변 부분을 찾아서 labels를 구성한다. 
        ########################################################
        assistant_header = "<|start_header_id|>assistant<|end_header_id|>\n"
        assistant_tokens = tokenizer.encode(assistant_header, add_special_tokens=False)  # assistant_header 토큰 index로 변환

        eot_token = "<|eot_id|>"
        eot_tokens = tokenizer.encode(eot_token, add_special_tokens=False)

        i = 0
        while i <= len(input_ids) - len(assistant_tokens):
            if input_ids[i:i + len(assistant_tokens)] == assistant_tokens:
                start = i + len(assistant_tokens)
                end = start
                while end <= len(input_ids) - len(eot_tokens):
                    if input_ids[end:end + len(eot_tokens)] == eot_tokens:
                        break
                    end += 1
                for j in range(start, end): # assistant 토큰(정답)을 -100에서 정답토큰값으로 변환
                    labels[j] = input_ids[j]
                for j in range(end, end + len(eot_tokens)): # eot 토큰을 다시 -100으로 변환
                    labels[j] = input_ids[j]
                break
            i += 1
        
        ##################################################################
        # 생성된 input_ids, attention_mask, labels를 new_batch에 추가한다.
        ##################################################################
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)
    

    ######################################################
    #  패딩 처리 
    #  -  배치내 입력중 가장 긴 sample에 길이를 맞춘다.
    ######################################################
    max_length = max(len(ids) for ids in new_batch["input_ids"])            
    for i in range(len(new_batch["input_ids"])):
        pad_len = max_length - len(new_batch["input_ids"][i]) 
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * pad_len)
        new_batch["attention_mask"][i].extend([0] * pad_len)
        new_batch["labels"][i].extend([-100] * pad_len)

    # list -> torch.Tensor로 변환
    for k in new_batch:
        new_batch[k] = torch.tensor(new_batch[k])

    return new_batch

In [47]:
######################
# 확인
######################
example = [trainset[1000], trainset[12], trainset[10]]
batch = collate_fn(example)

print("batch:")
print("input_ids 크기:", batch["input_ids"].shape)
print("attention_mask 크기:", batch["attention_mask"].shape)
print("labels 크기:", batch["labels"].shape)

IndexError: Invalid key: 1000 is out of bounds for size 90

## SFTConfig 설정

### LoRA 설정

- LoRAConfig 주요 매개변수

| 매개변수                | 의미/역할                                         | 주요 옵션·예시                                                                                                             |
| ------------------- | --------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| **lora\_alpha**     | LoRA 어댑터의 학습 스케일 팩터(변화 강도). 값이 크면 학습 변화가 완만해짐 | 16, 32, 64 등                                                                                                         |
| **lora\_dropout**   | 어댑터에만 적용되는 드롭아웃 확률. 과적합 방지                    | 0.05, 0.1 등                                                                                                          |
| **r**               | LoRA 어댑터의 랭크(정보량/두께). 값이 크면 더 많은 정보, 메모리 사용↑  | 8, 16, 32 등                                                                                                          |
| **bias**            | 기존 모델의 bias 파라미터도 LoRA로 튜닝할지 여부               | "none"(권장), "all"                                                                                                    |
| **target\_modules** | LoRA를 어떤 레이어(부분)에 적용할지 지정                     | "q\_proj", "v\_proj", "o\_proj", "up\_proj" 등<br>(모델 구조에 따라 다름)                                                      |
| **task\_type**      | LoRA가 적용될 문제 유형(파인튜닝 목적)                      | "CAUSAL\_LM"(생성), "SEQ\_CLS"(분류),<br>"SEQ\_2\_SEQ\_LM"(번역/요약),<br>"TOKEN\_CLS"(토큰분류),<br>"QUESTION\_ANSWERING"(질의응답) |

In [ ]:
###############################
# Peft(LoRA) 어뎁터 설정
###############################

from peft import LoraConfig
from trl import SFTConfig

peft_config = LoraConfig(
        r=8,
        lora_alpha=32,   
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
)

### SFTConfig 설정
- 전체 학습관련 설정

### SFTConfig 주요 매개변수

| 매개변수                                | 설명                                                  | 주요 예시 값/옵션                       |
| ----------------------------------- | --------------------------------------------------- | ---------------------------------------       |
| **output\_dir**                     | 결과 모델/로그를 저장할 경로 또는 저장소 ID                          | `"./results"`                |
| **num\_train\_epochs**              | 전체 데이터를 몇 번 반복 학습할지(에포크 수)                          | `3`, `5`                    |
| **per\_device\_train\_batch\_size** | 각 GPU(디바이스)에서 한 번에 입력할 데이터 수(배치 크기)                 | `2`, `4`, `8`            |
| **gradient\_accumulation\_steps**   | 여러 미니배치를 모아 한 번에 업데이트(실질 배치 크기 키우기)                 | `1`, `2`, `4`        |
| **gradient\_checkpointing**         | 메모리 절약 기능(필요할 때만 중간 계산값 저장)                         | `True`, `False`            |
| **optim**                           | 최적화 알고리즘(학습 방법)                                     | `"adamw_torch_fused"`              |
| **logging\_steps**                  | 몇 step마다 로그를 출력할지                                   | `10`, `50`, `100`                   |
| **save\_strategy**                  | 모델 저장 방식(주기)                                        | `"steps"`, `"epoch"`                  |
| **save\_steps**                     | 몇 step마다 모델을 저장할지                                   | `50`, `100`                         |
| **bf16**                            | bfloat16 연산 사용(GPU 메모리 절약)                          | `True`, `False`                      |
| **learning\_rate**                  | 파라미터 업데이트 속도(학습률)                                   | `1e-4`, `5e-5`                   |
| **max\_grad\_norm**                 | 그래디언트 클리핑 임계값(학습 안정화)                               | `0.3`, `1.0`                  |
| **warmup\_ratio**                   | 워밍업 단계 비율(초기 학습률 천천히 증가)                            | `0.03`, `0.1`                |
| **lr\_scheduler\_type**             | 학습률 조정 방식                                           | `"constant"`, `"linear"`               |
| **push\_to\_hub**                   | 학습 결과를 Hugging Face Hub로 업로드할지 여부                   | `True`, `False`                  |
| **hub\_model\_id**                  | 업로드할 Hugging Face Hub 저장소 ID                        |                                        |
| **hub\_token**                      | Hugging Face Hub 인증 토큰 사용 여부                        | `True`                                 |
| **remove\_unused\_columns**         | 학습에 안 쓰는 데이터 컬럼 자동 제거 여부                            | `True`, `False`               |
| **dataset\_kwargs**                 | 데이터셋 추가 옵션(딕셔너리 형태)                                 | `{"skip_prepare_dataset": True}` |
| **report\_to**                      | 학습 로그를 기록할 대상(예: wandb, tensorboard, 빈 리스트면 기록 안 함) | `None`, `["wandb"]`        |
| **max\_seq\_length**                | 한 입력에 허용되는 최대 토큰(단어) 수                              | `2048`, `4096`, `8192`          |
| **label\_names**                    | Trainer가 label로 인식할 컬럼명                             | `["labels"]`                           |


In [ ]:
epochs = 10

model_id = "kanana-1.5-2.1b-instruct-2505-finace_news-finetuning"
args = SFTConfig(
    output_dir=model_id,
    num_train_epochs=epochs,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4, 
    gradient_checkpointing=True,
    optim="adamw_torch_fused", 
    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    bf16=True,
    learning_rate=1e-4,
    max_grad_norm=0.3, 
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    
    push_to_hub=True,
    hub_model_id=f"kgmyh/{model_id}",
    hub_token=True,
    
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=max_seq_length,
    label_names=["labels"],
    report_to=None
)

## 학습하기

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=trainset,
    data_collator=collate_fn,
    peft_config=peft_config
)

In [ ]:
#############################
#  학습 시작
#############################
trainer.train()

In [ ]:
#############################
# 모델 저장
#############################
trainer.save_model()

In [ ]:
####################################################################################################################
#  수동으로 huggingface hub에 올리기
#  - 모델과 토크나이저를 같이 올린다.
#  - 학습도중 업로드되도록 했어도 토크나이저는 upload 되지 않기 때문에 수동으로 올려준다.
####################################################################################################################
trainer.push_to_hub()
tokenizer.push_to_hub("kanana-1.5-2.1b-instruct-2505-finace_news-finetuning")

# 테스트

## LoRA 파인튜닝 모델 사용법

- 위 과정에서 LoRA(Low-Rank Adaptation) 어댑터를 파인튜닝하여 Hugging Face Hub에 업로드한 상태
- LoRA는 전체 모델을 학습하는 대신 작은 어댑터만 학습하는 효율적인 방법

- **사용 시 필요한 구성 요소**
    - Base Model (기본 모델)
    - LoRA Adapter (파인튜닝된 어댑터)
    - 이 둘을 merge해서 사용한다. 
- **사용 방법 2가지**
    1. **개별 로드 후 병합**
        - 기본 모델과 LoRA 어댑터를 각각 다운로드
        - 런타임에서 두 구성 요소를 병합하여 사용

    2. **병합된 모델 직접 사용**
        - 미리 병합된 상태의 모델을 다운로드
        - 바로 사용 가능
    - 첫 번째 방법은 메모리 효율성과 유연성을 제공하고, 두 번째 방법은 사용 편의성을 제공합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "kakaocorp/kanana-1.5-2.1b-instruct-2505"
lora_model_id = "kgmyh/kanana-1.5-2.1b-instruct-2505-finace_news-finetuning"


tokenizer = AutoTokenizer.from_pretrained(base_model_id)
llama_input_creator = InputPromptCreator(tokenizer)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

In [ ]:
news_title = "\"분위기 좀 좋아지나 했는데...\" 삼성전자·SK하이닉스 또 애태운다"
news_document = """미 도널드 트럼프 행정부가 중국 내 반도체 공장에 미국산 장비 반입을 제한하겠다는 입장을 전달한 것으로 알려졌는데 중국에 생산 기지를 둔 삼성전자와 SK하이닉스도 긴장하고 있다. 이대로라면 새 반도체 장비를 중국으로 들여올 때마다 미 정부 승인을 받아야 하기에 중국 공장 운영에 차질을 빚을 수 있다는 걱정이 나온다.
22일 전자업계에 따르면 미 월스트리트저널(WSJ)은 20일 미 상무부가 대표적 메모리반도체 제조사인 삼성전자·SK하이닉스와 세계 파운드리(반도체 위탁생산) 1위 기업인 대만 TSMC에 미국서 만든 반도체 제조 장비의 중국에 공급할 때 줬던 '사전 허가 면제 조치'를 철회할 수 있다고 했다고 보도했다. 삼성전자와 SK하이닉스는 입장을 내놓지 않은 채 상황을 지켜보고 있다.
2022년 조 바이든 미국 행정부는 미국산 장비의 중국 반입을 규제하는 조치를 발표했지만 삼성전자·하이닉스·TSMC 등이 운영하던 공장에는 2023년 검증된 최종 사용자(VEU) 자격을 줘 사실상 이 규제를 무기한 유예했다. 이 때문에 삼성전자와 하이닉스는 허가 없이도 장비를 중국에 들여보냈다. 새 방침은 확정된 것은 아니고 미국 정부 내에서도 의견이 엇갈린 것으로 전해졌다.
VEU 자격 철폐설에 정부도 "업계 우려 전달"
도널드 트럼프(왼쪽) 미국 대통령과 키어 스타머 영국 총리가 16일 캐나다 앨버타주 캐내내스키스에서 열린 주요 7개국(G7) 회의 도중 양국 간 무역협정 서명문을 들고 기념촬영을 하고 있다. 캐내내스키스=로이터 연합
도널드 트럼프(왼쪽) 미국 대통령과 키어 스타머 영국 총리가 16일 캐나다 앨버타주 캐내내스키스에서 열린 주요 7개국(G7) 회의 도중 양국 간 무역협정 서명문을 들고 기념촬영을 하고 있다. 캐내내스키스=로이터 연합뉴스
삼성전자는 중국 시안에 낸드플래시 메모리 공장이, 쑤저우에는 후공정(패키징) 공장이 있다. SK하이닉스는 우시에서 D램 공장, 충칭에서 후공정 공장, 다롄에서 인텔로부터 인수한 낸드플래시 공장을 가동 중이다. 트럼프 정부가 최첨단 극자외선(EUV) 노광 장비의 중국 내 반입은 VEU와 상관없이 통제하고 있어 이들 공장은 최신 제품보다 범용(레거시) 반도체 공급에 활용된다.
업계에서는 실제 규제가 생기면 당장은 아니더라도 장기적으로는 공장 운영에 차질을 빚을 수 있다고 본다. 한 관계자는 "공정을 점검하고 고장이 나면 새 장비를 들여야 한다"면서 "중국 공장 운영에 불확실성이 생기는 것"이라고 설명했다.
여한구 산업자원부 통상교섭본부장은 22일 미국 측과 관세 협상을 위해 출국을 앞두고 "미 상무부와 무역대표부, 백악관 쪽과 접촉해 우리 업계의 우려 사항을 충분히 전달할 것"이라며 "건설적으로 협의해 나갈 부분이 있는지 최대한 신경 쓰겠다"고 말했다."""

message = llama_input_creator.create_pipeline_prompt(news_title=news_title, news_document=news_document)

In [ ]:
response = pipe(message, max_new_tokens=1000)

In [ ]:
print(response[0]['generated_text'][-1]["content"])

In [ ]:
#####################################
# Merge된 모델 한번에 받아오기
#####################################

from peft import AutoPeftModelForCausalLM
fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(lora_model_id, device_map="auto", torch_dtype=torch.bfloat16)
pipe2 = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

In [ ]:
response2 = pipe2(message, max_new_tokens=1000)

In [ ]:
print(response[0]['generated_text'][-1]["content"])

In [ ]:
####################################
#  기반 모델로 테스트
####################################

base_pipe = pipeline(task='text-generation', model=base_model, tokenizer=tokenizer)
response_base = base_pipe(message)

In [ ]:
print(response_base[0]['generated_text'][-1]["content"])